[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmaruyiyang/MMAI-MemeSonic/blob/main/image-text-fusion/colearning_audio.ipynb)

# Co-learning with Audio as Secondary Modality

**Idea** (Pham et al., AAAI 2019 — cyclic translation paradigm):
use a *secondary modality* (audio) only **during training** to improve the *primary* modality pair (image + text).
At test time, audio is dropped entirely — zero extra cost.

### Setup
| | Training | Test |
|---|---|---|
| **Input** | image + text + audio | image + text only |
| **Loss** | `L_ce` (classification) + `λ·L_align` (InfoNCE) | — |
| **Audio role** | supervision signal via contrastive alignment | dropped |

`L_align` pulls the fused image-text `align` vector toward the ImageBind audio embedding of the same meme, forcing the fusion to learn affectively richer representations.

### Dependencies on prior notebooks
- **`incongruity_aware_fusion.ipynb`** must have been run first to produce:
  - `artifacts/clip_concat_emb.npz` — cached CLIP embeddings for all splits
- **`audio/Audio_Gen_Trimodal_Align.ipynb`** must have been run to produce:
  - `<AUDIO_DIR>/<id>_script.mp3` — TTS audio for the 100 labeled samples
  - `<AUDIO_IDS_CSV>` — metadata CSV with `id` and `matched_ds_index` columns

## 0. Install & Config

In [ ]:
!pip install -q torch torchvision
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q datasets scikit-learn

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, recall_score, classification_report
from datasets import load_dataset

# ── Paths ─────────────────────────────────────────────────────────────────────
# Audio files produced by Audio_Gen_Trimodal_Align.ipynb
AUDIO_DIR     = '/content/drive/MyDrive/met_meme_audio_project/audio_script'
AUDIO_EXT     = '.mp3'
AUDIO_IDS_CSV = '/content/drive/MyDrive/met_meme_audio_project/results_100_samples_rescripted_en.csv'

# CLIP embedding cache from incongruity_aware_fusion.ipynb
CLIP_EMB_CACHE  = 'artifacts/clip_concat_emb.npz'

# Output artifacts
os.makedirs('artifacts', exist_ok=True)
AUDIO_EMB_CACHE = 'artifacts/audio_imagebind_emb.pt'
CKPT_BASELINE   = 'artifacts/colearn_baseline.pt'
CKPT_COLEARN    = 'artifacts/colearn_audio.pt'

# ── Hyperparameters ───────────────────────────────────────────────────────────
LAMBDA_ALIGN = 0.5    # co-learning loss weight; sweep [0.1, 0.3, 0.5, 1.0]
TEMP         = 0.07   # InfoNCE temperature
EPOCHS       = 10
LR           = 5e-4
WEIGHT_DECAY = 1e-3
BATCH_SIZE   = 64
NUM_CLASSES  = 5      # intention (Interactive/Expressive/Entertaining/Offensive/Other)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'LAMBDA_ALIGN={LAMBDA_ALIGN}  TEMP={TEMP}  EPOCHS={EPOCHS}')

## 1. Pre-extract ImageBind Audio Embeddings

Run once; result cached to `artifacts/audio_imagebind_emb.pt`.

In [ ]:
import pandas as pd

def extract_audio_embeddings(csv_path, audio_dir, ext, device):
    from imagebind import data as ib_data
    from imagebind.models import imagebind_model
    from imagebind.models.imagebind_model import ModalityType

    df = pd.read_csv(csv_path)
    ib_model = imagebind_model.imagebind_huge(pretrained=True).eval().to(device)

    paths, ids = [], []
    for _, row in df.iterrows():
        p = f"{audio_dir}/{int(row['id']):03d}_script{ext}"
        if os.path.exists(p):
            paths.append(p)
            ids.append(int(row['id']))

    print(f'Found {len(paths)} audio files')
    audio_input = ib_data.load_and_transform_audio_data(paths, device)
    with torch.no_grad():
        emb = ib_model({ModalityType.AUDIO: audio_input})[ModalityType.AUDIO]  # (N, 1024)
    emb = F.normalize(emb, dim=-1).cpu()
    del ib_model
    torch.cuda.empty_cache()
    return emb, ids


if os.path.exists(AUDIO_EMB_CACHE):
    print('Loading cached audio embeddings...')
    saved = torch.load(AUDIO_EMB_CACHE)
    audio_embs, audio_ids = saved['emb'], saved['ids']
else:
    print('Extracting ImageBind audio embeddings (runs once)...')
    audio_embs, audio_ids = extract_audio_embeddings(
        AUDIO_IDS_CSV, AUDIO_DIR, AUDIO_EXT, device)
    torch.save({'emb': audio_embs, 'ids': audio_ids}, AUDIO_EMB_CACHE)
    print(f'Saved → {AUDIO_EMB_CACHE}')

id_to_audioemb = {aid: audio_embs[i] for i, aid in enumerate(audio_ids)}
print(f'Audio embeddings: {audio_embs.shape}  ({len(audio_ids)} samples)')

## 2. Load CLIP Embeddings

Reuses the cache produced by `incongruity_aware_fusion.ipynb` (Section 6.4).
If the cache doesn't exist, this cell re-computes it.

In [ ]:
import clip
from torch.utils.data import Dataset

hf = load_dataset('Emmaruyi/met-meme-filtered')
clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()

class CLIPDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data[idx]
        img = clip_preprocess(row['image'].convert('RGB'))
        txt = clip.tokenize([row['caption']], truncate=True).squeeze(0)
        lbl = row['intention_num'] - 1
        return img, txt, lbl

def cache_clip_embeddings(hf_split):
    all_img, all_txt, all_lbl = [], [], []
    loader = DataLoader(CLIPDataset(hf_split), batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for imgs, texts, labels in loader:
            imgs, texts = imgs.to(device), texts.to(device)
            i_feat = F.normalize(clip_model.encode_image(imgs).float(), dim=-1).cpu()
            t_feat = F.normalize(clip_model.encode_text(texts).float(), dim=-1).cpu()
            all_img.append(i_feat); all_txt.append(t_feat); all_lbl.append(labels)
    return torch.cat(all_img), torch.cat(all_txt), torch.cat(all_lbl)

if os.path.exists(CLIP_EMB_CACHE):
    print('Loading cached CLIP embeddings...')
    d = np.load(CLIP_EMB_CACHE)
    tr_img = torch.from_numpy(d['tr_img']); tr_txt = torch.from_numpy(d['tr_txt']); tr_lbl = torch.from_numpy(d['tr_lbl'])
    va_img = torch.from_numpy(d['va_img']); va_txt = torch.from_numpy(d['va_txt']); va_lbl = torch.from_numpy(d['va_lbl'])
    te_img = torch.from_numpy(d['te_img']); te_txt = torch.from_numpy(d['te_txt']); te_lbl = torch.from_numpy(d['te_lbl'])
else:
    print('Caching CLIP embeddings for all splits...')
    tr_img, tr_txt, tr_lbl = cache_clip_embeddings(hf['train'])
    va_img, va_txt, va_lbl = cache_clip_embeddings(hf['validation'])
    te_img, te_txt, te_lbl = cache_clip_embeddings(hf['test'])
    np.savez(CLIP_EMB_CACHE,
             tr_img=tr_img.numpy(), tr_txt=tr_txt.numpy(), tr_lbl=tr_lbl.numpy(),
             va_img=va_img.numpy(), va_txt=va_txt.numpy(), va_lbl=va_lbl.numpy(),
             te_img=te_img.numpy(), te_txt=te_txt.numpy(), te_lbl=te_lbl.numpy())
    print(f'Saved → {CLIP_EMB_CACHE}')

print(f'Train: {len(tr_lbl)}  Val: {len(va_lbl)}  Test: {len(te_lbl)}')

## 3. Build Audio-Labeled Subset Index

The 100 audio samples were drawn from `hf['train']` with `matched_ds_index` recording the original HF row index.
We build a mapping `hf_train_idx → audio_emb` used by the co-learning loss.

In [ ]:
meta_df = pd.read_csv(AUDIO_IDS_CSV)

# matched_ds_index = row index in hf['train']; id = sequential 0-99 sample id
hf_train_idx_to_audio = {}
for _, row in meta_df.iterrows():
    hf_idx = int(row['matched_ds_index']) if 'matched_ds_index' in meta_df.columns else int(row['id'])
    aid    = int(row['id'])
    if aid in id_to_audioemb:
        hf_train_idx_to_audio[hf_idx] = id_to_audioemb[aid]

audio_index_list  = sorted(hf_train_idx_to_audio.keys())
audio_emb_tensor  = torch.stack([hf_train_idx_to_audio[i] for i in audio_index_list])  # (N_a, 1024)
audio_clip_img    = tr_img[audio_index_list]   # (N_a, 512) — CLIP image embs for audio rows
audio_clip_txt    = tr_txt[audio_index_list]   # (N_a, 512)

print(f'Train samples with audio: {len(audio_index_list)} / {len(tr_lbl)}')
print(f'audio_emb_tensor: {audio_emb_tensor.shape}')

## 4. Model Definition

- **`CLIPProjection`** — CLIP 512-dim → 64-dim (trainable; backbone frozen)
- **`SentimentIncongruityFusion`** — unchanged from `incongruity_aware_fusion.ipynb`
- **`ImageTextHead`** — MLP classifier on the 69-dim fused vector
- **`AudioAlignHead`** — projects ImageBind 1024-dim audio emb → 64-dim align space; **training only**, dropped at test time

In [ ]:
class CLIPProjection(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_proj = nn.Linear(512, 64)
        self.txt_proj = nn.Sequential(nn.Linear(512, 64), nn.ReLU())

    def forward(self, img_feat, txt_feat):
        return self.img_proj(img_feat), self.txt_proj(txt_feat)


class SentimentIncongruityFusion(nn.Module):
    """Unchanged from incongruity_aware_fusion.ipynb."""
    def __init__(self, input_dim=64, num_sentiments=5):
        super().__init__()
        self.img_sentiment = nn.Linear(input_dim, num_sentiments)
        self.txt_sentiment = nn.Linear(input_dim, num_sentiments)
        self.incong_gate   = nn.Sequential(nn.Linear(input_dim * 2, 1), nn.Sigmoid())

    def forward(self, img, txt):
        img_sent = F.softmax(self.img_sentiment(img), dim=-1)
        txt_sent = F.softmax(self.txt_sentiment(txt), dim=-1)
        incong   = torch.abs(img_sent - txt_sent)
        gate     = self.incong_gate(torch.cat([img, txt], dim=-1))
        align    = (img + txt) / 2
        fused    = torch.cat([align, gate * incong], dim=-1)  # (B, 69)
        return fused, align  # align returned separately for audio supervision


class ImageTextHead(nn.Module):
    def __init__(self, in_dim=69, hidden=256, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x): return self.net(x)


class AudioAlignHead(nn.Module):
    """Maps ImageBind 1024-dim audio emb → 64-dim align space.
    Used only during training; dropped at test time."""
    def __init__(self, audio_dim=1024, align_dim=64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(audio_dim, 256), nn.ReLU(),
            nn.Linear(256, align_dim)
        )
    def forward(self, x): return F.normalize(self.proj(x), dim=-1)


def build_model():
    return (
        CLIPProjection().to(device),
        SentimentIncongruityFusion(input_dim=64, num_sentiments=NUM_CLASSES).to(device),
        ImageTextHead(in_dim=69, num_classes=NUM_CLASSES).to(device),
        AudioAlignHead().to(device),
    )

print('Models defined.')
# Trainable param count (image-text path only, no audio head)
_cp, _fu, _cl, _ = build_model()
n_params = sum(p.numel() for m in [_cp, _fu, _cl] for p in m.parameters() if p.requires_grad)
print(f'Image-text path trainable params: {n_params:,}')

## 5. InfoNCE Alignment Loss

Symmetric cross-entropy over cosine similarity matrix between audio and fused align vectors.
Only called on the mini-batch of audio-labeled samples.

In [ ]:
def infonce_loss(audio_proj, align_proj, temperature=TEMP):
    """
    audio_proj : (B, D) L2-normalized
    align_proj : (B, D) L2-normalized
    """
    logits = audio_proj @ align_proj.T / temperature  # (B, B)
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_a = F.cross_entropy(logits,   labels)
    loss_b = F.cross_entropy(logits.T, labels)
    return (loss_a + loss_b) / 2

print('InfoNCE loss defined.')

## 6. Training Loop

In [ ]:
def train_and_eval(use_audio_loss, ckpt_path, label, lam=LAMBDA_ALIGN):
    """
    use_audio_loss=False → standard image-text fusion (baseline)
    use_audio_loss=True  → co-learning with audio alignment
    Returns (test_acc, macro_recall, history_dict)
    """
    clip_proj, fusion, cls_head, audio_head = build_model()

    params = (list(clip_proj.parameters()) +
              list(fusion.parameters()) +
              list(cls_head.parameters()))
    if use_audio_loss:
        params += list(audio_head.parameters())

    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

    tr_loader = DataLoader(TensorDataset(tr_img, tr_txt, tr_lbl),
                           batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(va_img, va_txt, va_lbl), batch_size=BATCH_SIZE)
    te_loader = DataLoader(TensorDataset(te_img, te_txt, te_lbl), batch_size=BATCH_SIZE)

    history = {'train_ce': [], 'train_align': [], 'val_acc': []}
    best_val, best_state = 0., None
    t0 = time.time()

    for epoch in range(EPOCHS):
        clip_proj.train(); fusion.train(); cls_head.train()
        if use_audio_loss: audio_head.train()
        ce_sum, al_sum, n_steps = 0., 0., 0

        for img_b, txt_b, lbl_b in tr_loader:
            img_f, txt_f = clip_proj(img_b.to(device), txt_b.to(device))
            fused, align = fusion(img_f, txt_f)
            logits = cls_head(fused)
            l_ce   = F.cross_entropy(logits, lbl_b.long().to(device))

            l_align = torch.tensor(0., device=device)
            if use_audio_loss and len(audio_emb_tensor) > 1:
                # Use the full audio subset every step (100 samples, lightweight)
                aud_proj   = audio_head(audio_emb_tensor.to(device))          # (N_a, 64)
                a_img, a_txt = clip_proj(
                    audio_clip_img.to(device), audio_clip_txt.to(device))
                _, a_align = fusion(a_img, a_txt)                              # (N_a, 64)
                align_norm = F.normalize(a_align, dim=-1)
                l_align    = infonce_loss(aud_proj, align_norm)

            loss = l_ce + lam * l_align
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ce_sum += l_ce.item(); al_sum += l_align.item(); n_steps += 1

        history['train_ce'].append(ce_sum / n_steps)
        history['train_align'].append(al_sum / n_steps)

        # ── Validation ────────────────────────────────────────────────────────
        clip_proj.eval(); fusion.eval(); cls_head.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for img_b, txt_b, lbl_b in va_loader:
                img_f, txt_f = clip_proj(img_b.to(device), txt_b.to(device))
                fused, _     = fusion(img_f, txt_f)
                preds        = cls_head(fused).argmax(1)
                correct += (preds == lbl_b.to(device)).sum().item()
                total   += lbl_b.size(0)
        val_acc = correct / total
        history['val_acc'].append(val_acc)

        elapsed = time.time() - t0
        print(f'[{label}] Epoch {epoch+1:02d}/{EPOCHS} | '
              f'CE={history["train_ce"][-1]:.4f} | '
              f'Align={history["train_align"][-1]:.4f} | '
              f'Val={val_acc:.4f} | {elapsed:.0f}s')

        if val_acc > best_val:
            best_val  = val_acc
            best_state = {
                'clip_proj': {k: v.clone() for k, v in clip_proj.state_dict().items()},
                'fusion':    {k: v.clone() for k, v in fusion.state_dict().items()},
                'cls_head':  {k: v.clone() for k, v in cls_head.state_dict().items()},
            }

    train_time = time.time() - t0
    print(f'\n[{label}] Training done in {train_time:.1f}s  ({train_time/EPOCHS:.1f}s/epoch)')

    # ── Load best val checkpoint ───────────────────────────────────────────────
    clip_proj.load_state_dict(best_state['clip_proj'])
    fusion.load_state_dict(best_state['fusion'])
    cls_head.load_state_dict(best_state['cls_head'])

    # ── Test (image+text only — audio head is never called here) ──────────────
    clip_proj.eval(); fusion.eval(); cls_head.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for img_b, txt_b, lbl_b in te_loader:
            img_f, txt_f = clip_proj(img_b.to(device), txt_b.to(device))
            fused, _     = fusion(img_f, txt_f)
            all_preds.append(cls_head(fused).argmax(1).cpu())
            all_true.append(lbl_b)
    all_preds = torch.cat(all_preds).numpy()
    all_true  = torch.cat(all_true).numpy()

    INTENTION_LABELS = ['Interactive', 'Expressive', 'Entertaining', 'Offensive', 'Other']
    acc    = accuracy_score(all_true, all_preds)
    recall = recall_score(all_true, all_preds, average='macro', zero_division=0)
    print(f'\n[{label}] Test Accuracy : {acc*100:.2f}%')
    print(f'[{label}] Macro Recall  : {recall*100:.2f}%')
    print(classification_report(all_true, all_preds,
          target_names=INTENTION_LABELS, zero_division=0))

    torch.save({'clip_proj': best_state['clip_proj'],
                'fusion':    best_state['fusion'],
                'cls_head':  best_state['cls_head'],
                'history':   history}, ckpt_path)
    print(f'Checkpoint saved → {ckpt_path}')
    return acc * 100, recall * 100, history

print('train_and_eval() defined.')

## 7. Run: Baseline vs Co-learning

In [ ]:
print('=' * 60)
print('CONDITION A: Image-Text Fusion — no audio (baseline)')
print('=' * 60)
acc_base, rec_base, hist_base = train_and_eval(
    use_audio_loss=False, ckpt_path=CKPT_BASELINE, label='Baseline')

In [ ]:
print('=' * 60)
print(f'CONDITION B: Co-learning with Audio  (λ={LAMBDA_ALIGN})')
print('=' * 60)
acc_colearn, rec_colearn, hist_colearn = train_and_eval(
    use_audio_loss=True, ckpt_path=CKPT_COLEARN, label='Co-learn')

## 8. λ Sweep (optional)

Sweeps `λ ∈ {0.1, 0.3, 0.5, 1.0}` to verify sensitivity of the co-learning effect.
Adds ~4× training time. Comment out if not needed.

In [ ]:
SWEEP_LAMBDAS = [0.1, 0.3, 0.5, 1.0]
sweep_results = {}  # lam -> (acc, recall)

for lam in SWEEP_LAMBDAS:
    print(f'\n── λ = {lam} ──')
    acc, rec, _ = train_and_eval(
        use_audio_loss=True,
        ckpt_path=f'artifacts/colearn_lam{lam}.pt',
        label=f'λ={lam}',
        lam=lam,
    )
    sweep_results[lam] = (acc, rec)

print('\n── λ sweep summary ──')
print(f'{"λ":>6}  {"Acc":>8}  {"MacroR":>8}')
for lam, (acc, rec) in sweep_results.items():
    print(f'{lam:>6.1f}  {acc:>7.2f}%  {rec:>7.2f}%')

## 9. Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── 9a. Val accuracy curves ───────────────────────────────────────────────────
for hist, label, c in [
    (hist_base,    'Baseline (no audio)',       'steelblue'),
    (hist_colearn, f'Co-learn (λ={LAMBDA_ALIGN})', 'crimson'),
]:
    axes[0].plot(range(1, EPOCHS + 1), [v * 100 for v in hist['val_acc']],
                 label=label, color=c, marker='o', markersize=4)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Accuracy (%)')
axes[0].set_title('Validation Accuracy')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# ── 9b. CE loss curves ────────────────────────────────────────────────────────
for hist, label, c in [
    (hist_base,    'Baseline', 'steelblue'),
    (hist_colearn, 'Co-learn', 'crimson'),
]:
    axes[1].plot(range(1, EPOCHS + 1), hist['train_ce'],
                 label=label, color=c, marker='o', markersize=4)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train CE Loss')
axes[1].set_title('Classification Loss')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# ── 9c. Test accuracy bar chart ───────────────────────────────────────────────
names  = ['Baseline\n(image+text)', f'Co-learn\n(+audio, λ={LAMBDA_ALIGN})']
accs   = [acc_base, acc_colearn]
colors = ['steelblue', 'crimson']
bars   = axes[2].bar(names, accs, color=colors, width=0.4,
                     edgecolor='black', linewidth=0.6)
for bar, acc in zip(bars, accs):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

delta = acc_colearn - acc_base
sign  = '+' if delta >= 0 else ''
axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title(f'Final Test Accuracy\n(Δ = {sign}{delta:.1f}%  |  test: image+text only)')
axes[2].set_ylim(0, max(accs) + 8)
axes[2].axhline(y=20, color='gray', linestyle='--', linewidth=0.8, label='Random (5-class)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle(
    'Co-learning with Audio as Secondary Modality\n'
    '(audio used only during training)',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('artifacts/colearning_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: artifacts/colearning_results.png')

print(f'\n{"="*50}')
print('SUMMARY')
print(f'{"="*50}')
print(f'Baseline  (image+text only) : {acc_base:.2f}%  Macro-R {rec_base:.2f}%')
print(f'Co-learn  (+audio training) : {acc_colearn:.2f}%  Macro-R {rec_colearn:.2f}%')
print(f'Delta                       : {sign}{delta:.2f}%')

## 10. Align Loss Curve (co-learning only)

Verifies the audio alignment loss actually decreases — if flat, the audio signal is not being absorbed.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, EPOCHS + 1), hist_colearn['train_align'],
        color='darkorange', marker='o', markersize=4, label=f'Align loss (λ={LAMBDA_ALIGN})')
ax.set_xlabel('Epoch'); ax.set_ylabel('InfoNCE Align Loss')
ax.set_title('Audio Alignment Loss During Training')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/align_loss_curve.png', dpi=150)
plt.show()
print('Saved: artifacts/align_loss_curve.png')